# Smart Attendance System (Exam Version)

In [21]:
!pip install opencv-python pandas matplotlib


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: c:\Users\Ohi\anaconda3\python.exe -m pip install --upgrade pip


In [22]:

import cv2, os

name = input("Student Name: ")

folder = os.path.join("student_images", name)
os.makedirs(folder, exist_ok=True)

cap = cv2.VideoCapture(0)
count = 0

print("Press S to save image, Q to quit")

while count < 20:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("Capture Faces", frame)

    key = cv2.waitKey(1)

    if key == ord('s'):
        cv2.imwrite(os.path.join(folder, f"{count}.jpg"), frame)
        count += 1
        print("Saved", count)

    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


Press S to save image, Q to quit


In [23]:

import cv2, os

detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

known_faces = {}

for student in os.listdir('student_images'):
    folder = os.path.join('student_images', student)
    samples = []

    for img_name in os.listdir(folder):
        img = cv2.imread(os.path.join(folder, img_name))
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        faces = detector.detectMultiScale(gray,1.2,5)

        for (x,y,w,h) in faces:
            face = gray[y:y+h,x:x+w]
            face = cv2.resize(face,(200,200))
            samples.append(face)

    known_faces[student] = samples

print("Students Loaded:", len(known_faces))


Students Loaded: 4


In [24]:

import pandas as pd, os

if not os.path.exists("attendance.csv"):
    pd.DataFrame(columns=["Name","Time"]).to_csv(
        "attendance.csv",
        index=False
    )


In [25]:

import cv2
import pandas as pd
import numpy as np
from datetime import datetime

cap = cv2.VideoCapture(0)
marked = set()

while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = detector.detectMultiScale(gray,1.2,5)

    for (x,y,w,h) in faces:

        face = gray[y:y+h,x:x+w]
        face = cv2.resize(face,(200,200))

        best_name = "Unknown"
        best_score = 999999

        for student, samples in known_faces.items():
            for sample in samples:

                score = np.mean(cv2.absdiff(face,sample))

                if score < best_score:
                    best_score = score
                    best_name = student

        cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)
        cv2.putText(frame,best_name,(x,y-10),
                    cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

        if best_name not in marked:
            marked.add(best_name)

            df = pd.read_csv("attendance.csv")
            df.loc[len(df)] = [best_name, str(datetime.now())]
            df.to_csv("attendance.csv", index=False)

    cv2.imshow("Attendance System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


In [26]:

import pandas as pd

attendance = pd.read_csv("attendance.csv")
print(attendance)
attendance


Empty DataFrame
Columns: [Name, Date, Time, Status]
Index: []


,Name,Date,Time,Status


In [27]:

import pandas as pd
import matplotlib.pyplot as plt

attendance = pd.read_csv("attendance.csv")

if len(attendance):
    attendance["Name"].value_counts().plot(kind="bar")
    plt.title("Attendance Summary")
    plt.show()
else:
    print("No attendance records found")


No attendance records found
